# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import logging

# Set logging level for PyTorch Tabular
logging.getLogger("pytorch_tabular").setLevel(logging.ERROR)

# Set logging level for PyTorch Lightning
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

# Set logging level for Lightning Fabric
logging.getLogger("lightning_fabric").setLevel(logging.ERROR)

# Disable device availability messages
logging.getLogger("lightning.pytorch.utilities.rank_zero").setLevel(logging.FATAL)

import pandas as pd
import numpy as np

from pytorch_tabular.models import CategoryEmbeddingModelConfig, TabNetModelConfig

import optuna

from functools import partial

from pathlib import Path
import sys

from typing import Type
from sklearn.base import BaseEstimator

from sklearn.metrics import (
    make_scorer,
    f1_score,
)

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    StatsModelsEstimator,
    PytorchTabularEstimator,
    deep_update_estimator_params,
    OptunaOptimizer,
    GridSearchOptimizer,
    RANDOM_STATE,
)

# from pytorch_lightning.loggers import MLFlowLogger
from pytorch_lightning.loggers import TensorBoardLogger

seed = RANDOM_STATE

# Fix error with save weights

# import torch
# # from omegaconf import DictConfig
# # from omegaconf.base import ContainerMetadata
# # import typing
# # torch.serialization.add_safe_globals([DictConfig, ContainerMetadata, typing.Any])
# import pytorch_tabular.utils.python_utils as pt_utils
# pt_utils.pl_load = lambda f, map_location: torch.load(f, map_location=map_location, weights_only=False)



In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings("ignore")

In [4]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())

True
1


# Settings

In [5]:
model_postfix = 'opt-model'

scoring_metrics = {"f1_macro": make_scorer(f1_score, average="macro"),}
step_scoring_average = "mean"
n_trials = 1000 # make 300

save_model_and_metrics = True
metrics_file = "metrics_modelling4_6-neural_networks.xlsx"

## Optimization function

In [6]:
def optimize_estimator_optuna(
    target:str,
    estimator:Type[BaseEstimator],
    estimator_params:dict,
    objective:callable,
    n_trials:int,
    add_smote:bool,
    is_smotenc:bool,
    smote_params:dict,
    step_scoring_average:str=step_scoring_average,
    params_to_process:list=None,
    model_postfix:str=model_postfix,
    scoring_metrics:dict=scoring_metrics,
    save_model_and_metrics:bool=save_model_and_metrics,
    metrics_file:str=metrics_file,
    opt_cv_folds:int=5,
    seed:int=seed,
):
    
    estimator_params = estimator_params.copy()
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        scoring_metrics=scoring_metrics,
        step_scoring_average=step_scoring_average,
        cv_folds=opt_cv_folds,
    )
    
    opt = OptunaOptimizer(
        objective=partial(objective, ml_pipe=ml_pipe),
        study_name="model_study",
        direction="maximize",
        seed=seed,
    )
    
    opt.optimize(n_trials=n_trials)
    
    best_params = opt.study.best_params
    
    print('raw best_params')
    display(best_params)
    
    if params_to_process:
        for param in params_to_process:
            if param == 'log2_size':
                keys = [k for k in best_params.keys() if param in k]
                sorted_keys = sorted(keys, key=lambda k: int(k.split("_")[1]))
                layer_sizes = [
                    2**best_params.pop(key)
                    for key in sorted_keys
                ]
                best_params['layers'] = '-'.join(map(str, layer_sizes))
            elif param == 'num_layers':
                key = next((k for k in best_params.keys() if param in k), None)
                if key:
                    best_params.pop(key)
            else: # Other cases
                # Find one key in best_params which contains param
                key = next((k for k in best_params.keys() if param in k), None)
                if key:
                    best_params[param] = best_params.pop(key)

    print('best_params')
    display(best_params)
    
    suggested_estimator_params = {
        "model_config_params": best_params
    }
    best_estimator_params = deep_update_estimator_params(
        ml_pipe=ml_pipe,
        suggested_params=suggested_estimator_params,
    )
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params={**estimator_params, **best_estimator_params},
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        step_scoring_average=step_scoring_average, # No need to pass, it would not be used
        metrics_file=metrics_file,
    )
    
    ml_pipe.run(
        save_model_and_metrics=save_model_and_metrics,
    )

## MultiLayer Perceptron (MLP)

In [7]:

def MLP_objective(
    trial:optuna.trial.Trial, 
    ml_pipe:MLPipeline,
):
    
    num_layers = trial.suggest_int('num_layers', 1, 3) # number of hidden layers
    layer_sizes = [
        2**trial.suggest_int(f'layer_{i}_log2_size', 0, 8)
        for i in range(num_layers)
    ]
    layers = '-'.join(map(str, layer_sizes))
    activation = trial.suggest_categorical('activation', ['ReLU', 'LeakyReLU', 'ELU'])
    dropout = trial.suggest_float('dropout', 0.0, 0.5)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
    use_batch_norm = trial.suggest_categorical('use_batch_norm', [True, False])
    batch_norm_continuous_input = trial.suggest_categorical('batch_norm_continuous_input', [True, False])
    
    suggested_estimator_params = {
        "model_config_params": {
            'layers': layers,
            'activation': activation,
            'dropout': dropout,
            'batch_norm_continuous_input': batch_norm_continuous_input,
            'use_batch_norm': use_batch_norm,
            'learning_rate': learning_rate,
        }
    }
    
    estimator_params = deep_update_estimator_params(
        ml_pipe=ml_pipe,
        suggested_params=suggested_estimator_params,
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score


In [8]:
estimator = PytorchTabularEstimator
target = "splashing"

add_smote = True
is_smotenc = False
smote_params = {
    # 'categorical_features': (
    #     'wettability',
    #     'inclination',
    # ),
}

params_to_process = [
    'num_layers', # drop this param
    'log2_size', # process to get string of layer sizes
]

estimator_params = {
    "model_class": CategoryEmbeddingModelConfig,
    # SET in optuna
    # "model_config_params": {
    #     'layers': '128-64-32',
    #     'activation': 'LeakyReLU',
    #     'dropout': 0.2,
    #     'batch_norm_continuous_input': True, # We already have normalized features    
    #     'use_batch_norm': False, # For hidden layers
    #     'learning_rate': 1e-3,
    # },
    # "data_config_params": {},
    "trainer_config_params": {
        'max_epochs': 300,
        'batch_size': 300, # let's condiser whole dataset
        'early_stopping': 'valid_loss',
        'early_stopping_patience': 10,
        'accelerator': 'gpu',
        'devices': 1,
        'progress_bar': 'none',
    },
    "optimizer_config_params": {
        'optimizer': 'Adam',
        'lr_scheduler': 'ReduceLROnPlateau',
        'lr_scheduler_params': {
            'patience': 5,
            'factor': 0.5,
        },
    },
}

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=MLP_objective,
    n_trials=n_trials,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-27 11:39:24,548] A new study created in memory with name: model_study
[I 2025-04-27 11:39:32,478] Trial 0 finished with value: 0.8360530192864537 and parameters: {'num_layers': 2, 'layer_0_log2_size': 8, 'layer_1_log2_size': 6, 'activation': 'ReLU', 'dropout': 0.02904180608409973, 'learning_rate': 0.029154431891537533, 'use_batch_norm': False, 'batch_norm_continuous_input': False}. Best is trial 0 with value: 0.8360530192864537.
[I 2025-04-27 11:40:27,233] Trial 1 finished with value: 0.78056709506644 and parameters: {'num_layers': 3, 'layer_0_log2_size': 1, 'layer_1_log2_size': 1, 'layer_2_log2_size': 1, 'activation': 'LeakyReLU', 'dropout': 0.14561457009902096, 'learning_rate': 0.0028016351587162596, 'use_batch_norm': False, 'batch_norm_continuous_input': False}. Best is trial 0 with value: 0.8360530192864537.
[I 2025-04-27 11:40:40,057] Trial 2 finished with value: 0.8332419940789144 and parameters: {'num_layers': 3, 'layer_0_log2_size': 1, 'layer_1_log2_size': 4, 'layer_

raw best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "PytorchTabularEstimator"


,0
target,splashing
model,CategoryEmbeddingModel_splashing_smote_opt-model
holdout_test_f1_macro,0.79
holdout_test_accuracy_balanced,0.78125
holdout_test_roc_auc,0.895833
holdout_test_f1,0.86
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.860788
cv_test_accuracy_balanced_median,0.873839
cv_test_roc_auc_median,0.931889


Model saved in ..\results\models_modelling4\CategoryEmbeddingModel_splashing_smote_opt-model


!!! Updated model name: `CategoryEmbeddingModel_splashing_smote_1000iters_opt-model`

Best choice:

- "activation": "LeakyReLU", (based on previous selection)
- "use_batch_norm": False,
- "batch_norm_continuous_input": True,

Params to optimize:
- dropout
- learning_rate
- layers